# HI-Small vs HI-Large — 9-Class 유형분류 비교 검토 노트북 (2026-09-02, 손은총)

같은 프로토콜로 두 세트를 나란히 놓는다: 공식 주 기간만(꼬리 절단 규칙 동일) · 시간순 60/20/20 · 같은 81피처(절대 시각 5열 제외) ·
같은 모델 설정(hier_et · 200트리 · balanced) · 운영점은 각 세트의 val 에서 재현율 0.70 고정 ·
블록 유형분류는 위상 35피처 ExtraTrees(attempt 학습, seed 5회) · seed-and-expand 는 W×m 격자를 각 세트의 val 로 선택.

**표본 변형**: 컨테이너 RAM 상한(cgroup 27.3 GiB) 때문에 HI-Large 는 train 전량(약 1억 행)을 못 올리고, random_r300(2,400만 행)도
ExtraTrees 학습 93분 만에 OOM 이라 **random_r100** 으로 학습했다. 같은 축 비교를 위해 HI-Small 도 random_r100 으로 재실행했다(`HI-Small(r100)`).
08-27 의 full 결과와 r300 은 참고 열로만 둔다.
HI-Large 전처리는 `--lowmem`(prep9/lowmem.py) — HI-Small 에서 원래 함수와 동치 검증(compare_lowmem_small.py)을 거쳤다.

표는 `make_compare_small_large.py` 가 두 산출물 트리에서 읽어 만든 CSV 다(재적합 없음). §5 는 test 재현율 0.70 지점을 알림 예산 곡선에서 직접 다시 찾는다.

In [1]:
import json, numpy as np, pandas as pd
from pathlib import Path
OUT = Path('/workspace/compare_small_large'); pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
A = pd.read_csv(OUT/'A_scale.csv'); B = pd.read_csv(OUT/'B_stage1_9class.csv'); C = pd.read_csv(OUT/'C_block_type.csv')
PC = pd.read_csv(OUT/'C_block_per_class.csv'); Dx = pd.read_csv(OUT/'D_seed_expand.csv'); R = json.load(open(OUT/'resources_large.json'))

## §1. 규모와 전처리 — 두 세트가 같은 규칙으로 잘렸는가

In [2]:
A.set_index('세트').T

세트,HI-Small(full),HI-Small(r300),HI-Small(r100),HI-Large(r100)
기간(일),10,10,10,97
행(정제 후),5076839,5076839,5076839,179639979
train/val/test,"3,045,987/1,015,195/1,015,657","3,045,987/1,015,195/1,015,657","3,045,987/1,015,195/1,015,657","107,783,265/35,928,623/35,928,091"
계좌,515078,515078,515078,2115733
8종 패턴 행,2554,2554,2554,113663
패턴 외 세탁,1968,1968,1968,87610
시도,363,363,363,16263
완결 시도,261,261,261,12177
채점자격 시도,105,105,105,5276
채택 창(분),10080,10080,10080,25920


## §2. 1차 9-Class — val 재현율 0.70 고정 운영점과 그 tau 를 test 에 적용한 결과

In [3]:
B.set_index('세트').T

세트,HI-Small(full),HI-Small(r300),HI-Small(r100),HI-Large(r100)
설정,hier_et·full·no_abs,hier_et·random_r300·no_abs,hier_et·random_r100·no_abs,hier_et·random_r100·no_abs
val K@R70,866,866,866,24657
val Pa@R70,0.576212,0.562356,0.56582,0.829095
val Pl@R70,0.58545,0.570439,0.571594,0.832015
val Pt@R70,0.182448,0.176674,0.174365,0.221154
val macroR8@R70,0.191186,0.180722,0.177529,0.136147
val P90 달성,True,True,False,True
val 최고 Pl,0.914286,0.95,0.891892,0.985
test 알림(val tau),843,838,805,24905
test Pa,0.56465,0.560859,0.575155,0.835214


## §3. 블록 단위 유형분류 — attempt(오라클) / window / pred(운영조건)

In [4]:
C

,세트,test_def,블록 수,블록 macro-F1(8),seed_sd,CI,거울쌍 6-class,간선 조건부 macro-F1,간선 n
0,HI-Small(full),attempt,183,0.650124,8.542731e-03,"[0.576,0.712]",0.659755,0.733211,684
1,HI-Small(full),window,232,0.499681,9.866527e-03,"[0.434,0.562]",0.483281,0.537376,684
2,HI-Small(full),pred,191,0.409294,5.527435e-03,"[0.339,0.472]",0.371906,0.452368,476
3,HI-Small(r300),attempt,183,0.650124,8.542731e-03,"[0.576,0.712]",0.659755,0.733211,684
4,HI-Small(r300),window,232,0.499681,9.866527e-03,"[0.434,0.562]",0.483281,0.537376,684
5,HI-Small(r300),pred,186,0.420362,2.542376e-03,"[0.348,0.480]",0.388240,0.458279,470
6,HI-Small(r100),attempt,183,0.650124,8.542731e-03,"[0.576,0.712]",0.659755,0.733211,684
7,HI-Small(r100),window,232,0.499681,9.866527e-03,"[0.434,0.562]",0.483281,0.537376,684
8,HI-Small(r100),pred,188,0.409340,5.551115e-17,"[0.338,0.469]",0.371963,0.447396,463
9,HI-Large(r100),attempt,7208,0.663803,2.897022e-04,"[0.654,0.674]",0.665721,0.736125,27431


In [5]:
t = PC[PC.test_def=='pred'].pivot(index='클래스', columns='세트', values=['support','정밀도','재현율','F1']).round(3)
t

support                                                         정밀도                                                         재현율                                                          F1  \
세트             HI-Large(r100) HI-Small(full) HI-Small(r100) HI-Small(r300) HI-Large(r100) HI-Small(full) HI-Small(r100) HI-Small(r300) HI-Large(r100) HI-Small(full) HI-Small(r100) HI-Small(r300) HI-Large(r100)   
클래스                                                                                                                                                                                                                 
BIPARTITE              1423.0           26.0           25.0           23.0          0.000          0.000          0.000          0.000          0.000          0.000          0.000          0.000          0.000   
CYCLE                   501.0           24.0           22.0           22.0          1.000          1.000          1.000          1.000          0.058          0.125          0.091          0.136          0.109   
FAN-IN                  463.0           15.0           15.0           15.0          0.550          0.480          0.500          0.500          0.631          0.800          0.800          0.867          0.588   
FAN-OUT                 487.0           15.0           15.0           14.0          0.498          0.588          0.556          0.588          0.628          0.667          0.667          0.714          0.556   
GATHER-SCATTER          726.0           28.0           28.0           28.0          0.441          0.500          0.529          0.500          0.543          0.321          0.321          0.321          0.486   
RANDOM                  382.0           11.0           10.0           11.0          0.073          0.116          0.114          0.125          0.654          0.909          0.900          0.909          0.131   
SCATTER-GATHER          509.0           17.0           17.0           17.0          0.523          0.889          1.000          1.000          0.246          0.471          0.412          0.412          0.334   
STACK                  2161.0           55.0           56.0           56.0          0.742          0.848          0.829          0.857          0.313          0.509          0.607          0.536          0.441   

                                                             
세트             HI-Small(full) HI-Small(r100) HI-Small(r300)  
클래스                                                          
BIPARTITE               0.000          0.000          0.000  
CYCLE                   0.222          0.167          0.240  
FAN-IN                  0.600          0.615          0.634  
FAN-OUT                 0.625          0.606          0.645  
GATHER-SCATTER          0.391          0.400          0.391  
RANDOM                  0.206          0.202          0.220  
SCATTER-GATHER          0.615          0.583          0.583  
STACK                   0.636          0.701          0.659

## §4. seed-and-expand — 각 세트 val 로 고른 설정과 기준의 차이

In [6]:
Dx.set_index('세트').T

세트,HI-Small(full),HI-Small(r300),HI-Small(r100),HI-Large(r100)
기준 val F1,0.261909,0.285921,0.281823,0.179908
기준 test F1,0.409294,0.420362,0.40934,0.215646
선택 W,10080,10080,10080,10080
선택 m,2.0,2.0,1.0,1.5
선택 val F1,0.321233,0.299367,0.281823,0.180301
선택 test F1,0.436301,0.422625,0.40934,0.213783
선택 test CI,"[0.368,0.495]","[0.347,0.483]","[0.340,0.469]","[0.206,0.222]"
기준 CYCLE R,0.125,0.136364,0.090909,0.007776
선택 CYCLE R,0.142857,0.15,0.090909,0.007849
기준 멤버정밀도,0.672261,0.665771,0.681734,0.867589


## §5. 라이브 재현 — test 알림 예산 곡선에서 재현율 0.70 지점을 직접 찾는다 (표 B 의 test K@R70 과 같아야 함)

In [7]:
for name, mdir in (('HI-Small(full)','/workspace/model_9class'), ('HI-Small(r300)','/workspace/model_9class_r300'), ('HI-Small(r100)','/workspace/model_9class_r100'), ('HI-Large(r100)','/workspace/model_9class_large')):
    p = Path(mdir)/'tables/alert_budget_sweep_test.csv'
    if not p.exists(): print(name, '산출물 없음'); continue
    sw = pd.read_csv(p); ok = sw[sw['recall_pattern'] >= 0.70]
    r = ok.sort_values('k_alerts').iloc[0] if len(ok) else None
    print(name, '→ test R≥0.70 최소 K:', None if r is None else f"K={int(r['k_alerts'])} Pa={r['precision_any']:.4f} Pl={r['precision_laundering']:.4f}")

HI-Small(full) → test R≥0.70 최소 K: K=866 Pa=0.5554 Pl=0.5681
HI-Small(r300) → test R≥0.70 최소 K: K=1067 Pa=0.4789 Pl=0.4958
HI-Small(r100) → test R≥0.70 최소 K: K=1067 Pa=0.4686 Pl=0.4845
HI-Large(r100) → test R≥0.70 최소 K: K=24657 Pa=0.8387 Pl=0.8412


## §6. 자원 (Large 체인, bash 측정: elapsed / peak RSS VmHWM)

In [8]:
pd.DataFrame(R).T

,elapsed_s,peak_rss_GiB
blocks,390,14
expand,750,12
final_blocks,121,0
prep,2551,19
